# Module 3: LangSmith — Observability & Evaluations

> Part of the **Modular Workshops** series. Standalone, ~25 min.

We cover three parts plus a closing loop:

1. **Tracing** — generate traces with the agent from Module 1, then query them with `runs.query` + filters.
2. **Offline evaluations** — build a dataset, then score it two ways: final-response (LLM-as-judge) and trajectory.
3. **Online evaluations** — score every new trace as it lands, via a UI-created evaluator.

Then **annotation queues** close the loop: route runs flagged by eval scores to a human for review.

<img src="../images/evals-conceptual.png" style="width: auto; max-height: 400px; border-radius: 8px;">

## Setup


In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model
from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)

import os, time
from datetime import datetime, timedelta, timezone
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
from langsmith import Client, uuid7

client = Client()
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "not set"))
print("Project:", os.environ.get("LANGSMITH_PROJECT", "default"))


LANGSMITH_TRACING: true
Project: modular-workshops


## Warm-up: Generate a few traces

Before we look at tracing and querying, let's actually produce some traces. 
We invoke the research agent from Module 1 (`agents/research_agent.py`) three times with **deliberately light** prompts — 
each one says "search at most once" so the runs finish in a few seconds instead of a few minutes.

On the trial run while building this module the warm-up took **~9 seconds total (3.1s avg per call)**. Expect similar.


In [2]:
from agents.research_agent import build_research_agent

agent = build_research_agent()

warmup_prompts = [
    "In one sentence, what is LangGraph for? Search at most once.",
    "In one sentence, what does the LangChain v1 release add? Search at most once.",
    "In one sentence, what is Tavily? Search at most once.",
]

total = 0.0
for q in warmup_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

print(f"\nTotal: {total:.1f}s ({total/len(warmup_prompts):.1f}s avg)")


[ 8.7s] LangGraph is a framework for building reliable, long-running, stateful workflows and AI agents that handle complex tasks
[12.7s] LangChain v1 adds a streamlined agent-focused API centered on `create_agent`, with middleware support and simplified acc
[ 8.4s] Tavily is a real-time web-search and content-extraction API built to provide AI agents and retrieval-augmented generatio

Total: 29.8s (9.9s avg)


## Part 1. Tracing + Querying Traces

Set `LANGSMITH_TRACING=true` and every LLM call, tool call, and state transition lands in your tracing project — no code changes required. 
(The warm-up above already produced traces; this section pulls them back out.)

We use `client.runs.query(...)` to query them.

In [3]:
project_name = os.environ.get("LANGSMITH_PROJECT", "modular-workshops")
try:
    project = client.read_project(project_name=project_name)
    print(f"Project: {project.name}")
    print(f"View traces: {project.url}")
except Exception as e:
    print(f"Could not read project (this is fine if first run): {e}")


Project: modular-workshops
View traces: https://smith.langchain.com/o/22e7329b-39d3-45f3-baf7-2ab127e18246/projects/p/a2ed48ab-d0e2-43b7-9d9a-3cfd309c11a7


### 1.1 Pull recent traces

`client.list_runs(...)` is deprecated in favor of `client.runs.query(...)` — async, and a few parameters changed shape. Useful ones:

- `project_ids=` — scope to one or more projects (resolve a name to an ID first with `client.read_project(...)`)
- `min_start_time=` / `max_start_time=` — time window (was `start_time=` / `end_time=`)
- `run_type=` — now uppercase: `"LLM"`, `"TOOL"`, `"CHAIN"`, `"RETRIEVER"`
- `has_error=True` — only failed runs (was `error=True`)
- `is_root=True` — only top-level traces (not their children) — unchanged
- `filter=` — LangSmith filter DSL (latency, feedback, attributes...) — syntax unchanged
- `selects=` — fields to return; defaults to `ID` only, so request what you need, e.g. `["NAME", "STATUS", "LATENCY_SECONDS", "ERROR"]`

There's no `limit=` — it returns an async iterable with cursor-based pagination. Iterate with `async for` and `break` once you have enough.

In [4]:
from datetime import datetime, timedelta, timezone

# Pull the last hour of root traces from this workshop's project
since = datetime.now(timezone.utc) - timedelta(hours=1)

recent_runs = []
async for r in client.runs.query(
    project_ids=[str(project.id)],
    min_start_time=since,
    is_root=True,
    selects=["ID", "NAME", "LATENCY_SECONDS", "ERROR"],
):
    recent_runs.append(r)
    if len(recent_runs) >= 20:
        break

print(f"Found {len(recent_runs)} root run(s) in the last hour\n")
for r in recent_runs[:5]:
    print(f"- {r.id}  {r.name:25s}  latency={r.latency_seconds}s  error={r.error is not None}")

Found 18 root run(s) in the last hour

- 01a19f7a-b616-7f21-b123-af26a6c1b8dd  LangGraph                  latency=Nones  error=False
- 01a19f7a-8464-7da3-b9f7-7cf28672f696  LangGraph                  latency=12.720063924789429s  error=False
- 01a19f7a-6253-73e1-a93b-f439badb5120  LangGraph                  latency=8.717458963394165s  error=False
- 01a19f6e-3e75-7180-aa99-98d81eb11662  LangGraph                  latency=8.42129921913147s  error=False
- 01a19f6e-1be2-75c0-9b3c-897bd22af9ff  LangGraph                  latency=8.849853992462158s  error=False


### 1.2 Filter DSL — find slow or errored runs

The `filter` argument is a small expression language. Common patterns:

- `gt(latency, 5)` — slower than 5 seconds
- `eq(status, "error")` — failed runs
- `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` — low-scored runs on a feedback key
- Combine with `and(...)` / `or(...)`

In [5]:
# Find slow root runs in the last hour (>5s latency)
slow_runs = []
async for r in client.runs.query(
    project_ids=[str(project.id)],
    min_start_time=since,
    is_root=True,
    filter='gt(latency, 5)',
    selects=["ID", "NAME", "LATENCY_SECONDS"],
):
    slow_runs.append(r)
    if len(slow_runs) >= 20:
        break

print(f"{len(slow_runs)} slow root run(s) (>5s) in the last hour")
for r in slow_runs[:5]:
    print(f"  {r.name:25s}  {r.latency_seconds:.1f}s  {r.id}")

17 slow root run(s) (>5s) in the last hour
  LangGraph                  8.4s  01a19f7a-b616-7f21-b123-af26a6c1b8dd
  LangGraph                  12.7s  01a19f7a-8464-7da3-b9f7-7cf28672f696
  LangGraph                  8.7s  01a19f7a-6253-73e1-a93b-f439badb5120
  LangGraph                  8.4s  01a19f6e-3e75-7180-aa99-98d81eb11662
  LangGraph                  8.8s  01a19f6e-1be2-75c0-9b3c-897bd22af9ff


## Part 2. Offline Evaluations

**Offline evals** are the experiments you run on demand against a fixed dataset.
Build a dataset once, score your agent against it whenever you change a prompt, a model, or a tool — get a clean before/after comparison.

Three pieces:
1. **Dataset** — labeled `inputs` + expected `outputs`
2. **Target function** — runs your agent on each example
3. **Evaluators** — score the output (LLM-as-judge or code-based)

The agent runs in this process; each example's input maps to a value you score — final-response and trajectory evals, below.

### 2.1 Dataset

Same input set, two reference shapes — one for final-response judging, one for trajectory matching.

In [6]:
examples = [
    {
        "inputs": {"query": "Write a haiku about Python to /haiku.txt"},
        "outputs": {
            "reference_answer": "A haiku about Python should be saved to /haiku.txt",
            "trajectory": ["write_file"],
        },
    },
    {
        "inputs": {"query": "Research the latest AI agent frameworks and write a brief report to /report.md"},
        "outputs": {
            "reference_answer": "A short report on recent AI agent frameworks, written to /report.md",
            "trajectory": ["task", "write_file"],
        },
    },
    {
        "inputs": {"query": "Write a short memo to /memo.md, then read it back to confirm"},
        "outputs": {
            "reference_answer": "A short memo written to /memo.md and read back",
            "trajectory": ["write_file", "read_file"],
        },
    },
    {
        "inputs": {"query": "Plan out a small research project on LangSmith, then research it and write a report to /report.md"},
        "outputs": {
            "reference_answer": "A planned-out research report on LangSmith written to /report.md",
            "trajectory": ["write_todos", "task", "write_file"],
        },
    },
]

dataset_name = "modular-workshops-evals"

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")


Deleted existing dataset 'modular-workshops-evals'
Created dataset 'modular-workshops-evals' with 4 examples
View: https://smith.langchain.com/o/22e7329b-39d3-45f3-baf7-2ab127e18246/datasets/b73cda0d-1b07-4c55-95cb-0c1fc0c9f29b


### 2.2 Final-response eval (LLM-as-judge)

<img src="../images/final-response.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Treat the agent as a black box: did the final response satisfy the request? 
We'll build the **LLM-as-judge from scratch** so the moving parts are clear:

1. A Pydantic / TypedDict schema for the judge's output (`score`, `reasoning`)
2. A judge prompt that explains the grading criteria
3. `model.with_structured_output(...)` to force the LLM into the schema
4. An evaluator function that calls the judge and returns the score in the shape `client.evaluate` expects

In [12]:
def run_agent_final(inputs: dict) -> dict:
    """Run the agent and return its response plus any files it wrote.

    The evaluator receives these as separate fields so it can inspect both
    the assistant's message and the resulting file artifacts.
    """
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )

    return {
        "response": result["messages"][-1].text,
        "files": result.get("files") or {},
    }

In [13]:
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage

# Define the judge's output schema -- with_structured_output enforces this shape on the LLM response.
class CorrectnessGrade(TypedDict):
    """Score whether the agent's response satisfied the user's request."""
    score: bool   # True if correct/helpful, False otherwise
    reasoning: str  # one-sentence explanation

# The dataset's `reference_answer` is a SUCCESS RUBRIC, not an expected response text.
# Make that explicit to the judge so it doesn't downscore valid agent responses that
# happen to be worded differently.
correctness_judge_prompt = """You are an expert grader evaluating an AI assistant.

You will see:
1. The user's request
2. The assistant's final response
3. Any files written by the assistant
4. A success rubric

Evaluate both the response and the files. If the task asks the assistant to write a file, inspect the file contents when available. Mark `score=True` if the task was completed successfully, even if the wording differs from the rubric.

Mark `score=False` only if the task was clearly missed, the file is missing or incorrect, the response contains errors, or the assistant refused without good reason.

Give one short sentence of reasoning.
"""

# Bind the schema once -- `judge` is now a structured-output LLM.
judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Files written: {outputs.get('files', {})}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])

    return {
        "key": "correctness",
        "score": int(grade["score"]),
        "comment": grade["reasoning"],
    }

In [14]:
results = client.evaluate(
    run_agent_final,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix="final-response",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


View the evaluation results for experiment: 'final-response-c3379a51' at:
https://smith.langchain.com/o/22e7329b-39d3-45f3-baf7-2ab127e18246/datasets/b73cda0d-1b07-4c55-95cb-0c1fc0c9f29b/compare?selectedSessions=f2b295db-44ba-4926-9e0a-7259911adf89




0it [00:00, ?it/s]

View at: final-response-c3379a51


### 2.3 Trajectory eval

<img src="../images/trajectory.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Score the **sequence of tool calls** the agent took, not just the final answer. Three evaluators:

- **`exact_match`** — did it take exactly the right steps in order?
- **`extra_steps`** — how many extra tool calls did it make?
- **`missing_steps`** — how many expected steps did it skip?

`extra_steps` and `missing_steps` use `collections.Counter` for multiset diffs — order doesn't matter for those two, but `exact_match` still catches ordering bugs.

In [15]:
from collections import Counter
from typing import Any

def trajectory_match(outputs, reference_outputs):
    return {
        "key": "exact_match",
        "score": int(outputs["trajectory"] == reference_outputs["trajectory"]),
    }

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs["trajectory"])
    return {"key": "missing_steps", "score": sum(missing.values())}


Next, we'll define the run function.

In [16]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Extract tool call names from messages in order."""
    tool_names = []
    for msg in messages:
        if getattr(msg, "tool_calls", None):
            tool_names.extend(tc["name"] for tc in msg.tool_calls)
    return tool_names

def run_agent_trajectory(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    return {"trajectory": extract_tool_calls(result["messages"])}


In [17]:
results = client.evaluate(
    run_agent_trajectory,
    data=dataset_name,
    evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix="trajectory",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


View the evaluation results for experiment: 'trajectory-ebc9d552' at:
https://smith.langchain.com/o/22e7329b-39d3-45f3-baf7-2ab127e18246/datasets/b73cda0d-1b07-4c55-95cb-0c1fc0c9f29b/compare?selectedSessions=be497970-070d-464d-95c3-fedfdd89d23a




0it [00:00, ?it/s]

View at: trajectory-ebc9d552


## Part 3. Online Evaluations

**Online evals** run automatically against every new trace as it lands in your tracing project — same judge logic as Part 2, just triggered on incoming runs instead of a dataset.

Set this up from the LangSmith UI:

1. Open your project in LangSmith (link below), then click the **Evaluators** tab.
2. Click **+ Evaluator** → **Create new**.
3. Give it a prompt and output schema — reuse the correctness judge from Part 2: score whether the response satisfied the request, as a `correctness` boolean plus a one-sentence `comment`.
4. Scope it to root traces only, set sampling to 100%, and save.

In [17]:
print(f"Open your project here, then click the Evaluators tab:\n{project.url}")

Open your project here, then click the Evaluators tab:
https://smith.langchain.com/o/22e7329b-39d3-45f3-baf7-2ab127e18246/projects/p/a2ed48ab-d0e2-43b7-9d9a-3cfd309c11a7


## Annotation Queues — Close the Loop

Once runs have **feedback scores** (from the evaluator you just set up in the UI, or any other source), route the low-scoring ones to a human for review.

LangSmith's annotation queues are that queue. We use **the same `create_run_rule` helper** — this time with `add_to_annotation_queue_id` set instead of an LLM judge. 
Any run matching the filter is added to the queue automatically.

In [18]:
queue = get_or_create_annotation_queue(
    client,
    name="modular-workshops-needs-review",
    description="Runs routed here by the workshop's correctness automation rule.",
)
print(f"Queue: {queue.name} (id={queue.id})")


Queue: modular-workshops-needs-review (id=393f60dc-4d1f-41c7-8622-df7e1d4f4343)


In [19]:
# Same helper, no evaluator this time -- just a routing rule.
# Filter: only root traces (is_root=true) with correctness > 0.5.
queue_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="workshop-route-correctness",
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "correctness"), '
        'gt(feedback_score, 0.5)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID:", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])


Queue rule ID: d3076950-3bb2-40ee-a97c-1d9d444b713b
Open in UI:     https://smith.langchain.com/o/22e7329b-39d3-45f3-baf7-2ab127e18246/projects/p/a2ed48ab-d0e2-43b7-9d9a-3cfd309c11a7?runview=threads&tab=2


### Trigger the evaluator and the queue rule

Both are live now. Run a few more light traces and you'll see:

1. The UI evaluator fires on each new trace and attaches a `correctness` feedback score (~30s delay).
2. The queue rule fires on each *new feedback* that matches its filter (low correctness) and routes the run to the review queue.

In [ ]:
trigger_prompts = [
    "In one sentence, what is LangSmith? Search at most once.",
    "In one sentence, what does `create_agent` do in LangChain v1? Search at most once.",
    "In one sentence, what is the difference between LangChain and LangGraph? Search at most once.",
]

time.sleep(100)

total = 0.0
for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

# Use the tenant_id LangSmith returned with the rule so the link works regardless of workspace.
tenant_id = queue_rule["payload"]["tenant_id"]

print(f"\nTotal: {total:.1f}s.")
print(f"\nProject (check the Evaluators tab): {project.url}")
print(f"Queue rule:        {queue_rule['url']}")
print(f"Queue (review UI): https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}")
print("\nFeedback shows up on traces within ~30s; queue placements follow once feedback lands.")

### Common run-rule patterns

Swap the `filter` to build different rules:

| Use Case | `filter` |
|---|---|
| Low online correctness | `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` |
| Errored runs | `eq(status, "error")` |
| Slow runs | `gt(latency, 10)` |
| Long-running tool calls | `and(eq(run_type, "tool"), gt(latency, 3))` |
| Specific tool fired | `eq(name, "tavily_search")` |

Both rule types — online eval and queue routing — go through the same `create_run_rule` helper. 
Use `delete_run_rule(client, rule_id)` to tear them down when you're done.


## Recap

| Part | What | API |
|---|---|---|
| **Warm-up** | Generate a few traces with the Module 1 agent | `agent.invoke(...)` |
| **1. Tracing + querying** | Auto-capture every run; pull back by filter | `LANGSMITH_TRACING=true`, `client.runs.query(filter=...)` |
| **2. Offline evals** | Score on demand against a dataset | `model.with_structured_output(...)` + `client.evaluate` |
| **3. Online evals** | Score every new trace automatically | LangSmith UI: project → Evaluators tab → + Evaluator |
| **Annotation queues** | Route flagged runs for human review | `create_run_rule(..., add_to_annotation_queue_id=...)` |

The full loop: trace → online eval scores it → run rule routes low scores to the queue → human reviews → fixes flow into the next dataset.

This closes the loop across all three modules: build the agent (Module 1), deploy it (Module 2), then observe, evaluate, and improve it in production (Module 3).